# Lab 5A: Exploring Data Without Persistence - SOLUTION

This notebook contains the solution for Lab 5A exercises.

## 💻 Exercise 1: Comprehensive File Format Comparison - SOLUTION

### Solution

In [ ]:
import duckdb
import time
import os
import pandas as pd

con = duckdb.connect(':memory:')

# Create a realistic dataset
con.execute("""
    CREATE TABLE sales_data AS
    SELECT 
        (random() * 1000)::int as customer_id,
        (random() * 100)::int as product_id,
        random() * 500 as amount,
        ['credit_card', 'debit_card', 'paypal'][((random() * 3)::int)] as payment_method,
        '2023-01-01'::date + (random() * 365)::int as sale_date
    FROM range(100000)
""")

# Test different formats
formats = ['csv', 'parquet', 'json']
results = []

for format in formats:
    # Export
    start = time.time()
    con.execute(f"COPY sales_data TO 'data/sales.{format}' (FORMAT {format.upper()})")
    export_time = time.time() - start
    
    # File size
    file_size = os.path.getsize(f'data/sales.{format}') / 1024  # KB
    
    # Query performance
    start = time.time()
    con.execute(f"SELECT COUNT(*), AVG(amount) FROM 'data/sales.{format}'").fetchone()
    query_time = time.time() - start
    
    results.append({
        'format': format,
        'export_time': export_time,
        'file_size': file_size,
        'query_time': query_time
    })

# Display results
print("File Format Comparison Results:")
print("="*60)
for result in results:
    print(f"\n{result['format'].upper()}:")
    print(f"  Export time: {result['export_time']:.4f}s")
    print(f"  File size: {result['file_size']:.2f}KB")
    print(f"  Query time: {result['query_time']:.4f}s")

# Analysis
print("\n" + "="*60)
print("Analysis:")
print("- Parquet offers best compression and query performance for analytics")
print("- CSV is human-readable but larger and slower for large datasets")
print("- JSON is flexible but inefficient for analytical workloads")
print("- Recommendation: Use Parquet for production analytics pipelines")

## 💻 Exercise 2: Complex JSON Shredding - SOLUTION

### Solution

In [ ]:
import json
import duckdb

con = duckdb.connect(':memory:')

# Create complex nested JSON
complex_data = {
    "company": "TechCorp",
    "departments": [
        {
            "name": "Engineering",
            "employees": [
                {"id": 1, "name": "Alice", "skills": ["Python", "SQL"], "salary": 90000},
                {"id": 2, "name": "Bob", "skills": ["Java", "Go"], "salary": 95000}
            ]
        },
        {
            "name": "Sales",
            "employees": [
                {"id": 3, "name": "Charlie", "skills": ["Sales", "CRM"], "salary": 70000}
            ]
        }
    ]
}

# Write to file
with open('data/complex_nested.json', 'w') as f:
    json.dump(complex_data, f)

# Query and flatten the nested structure
result = con.execute("""
    SELECT 
        departments[1]->>'name' as department_name,
        employees->>'name' as employee_name,
        employees->>'salary' as salary,
        len(employees->'skills') as skill_count
    FROM read_json('data/complex_nested.json', format='auto'),
         unnest(departments, recursive := true) AS t(departments)
         CROSS JOIN unnest(departments->'employees') AS t(employees)
""").fetchdf()

print("Flattened employee data from complex nested JSON:")
print(result)